In [4]:
from labjack import ljm
import time

# ── Configuration ──────────────────────────────────────────────────────────────
OUTPUT_REGISTER = "DAC1"
V_CLOSED        = 0.0     # Voltage corresponding to fully closed
V_OPEN          = 5.0     # Voltage corresponding to fully open
RAMP_STEPS      = 10      # Number of steps in each ramp
RAMP_DELAY_S    = 0.5     # Seconds between each ramp step
HOLD_TIME_S     = 120     # Seconds to hold at fully open before closing


def ramp(handle, v_start, v_end, steps, delay_s):
    """Ramp DAC0 linearly from v_start to v_end in `steps` increments."""
    for i in range(steps + 1):
        voltage = v_start + (v_end - v_start) * (i / steps)
        ljm.eWriteName(handle, OUTPUT_REGISTER, voltage)
        print(f"  DAC0 = {voltage:.2f} V")
        time.sleep(delay_s)


def main():
    handle = None
    try:
        # ── Connect ───────────────────────────────────────────────────────────
        handle = ljm.openS("T7", "ANY", "ANY")
        info   = ljm.getHandleInfo(handle)
        print(f"Connected to LabJack T7 [Serial: {info[2]}]")
        print(f"Output register: {OUTPUT_REGISTER}")
        print("-" * 40)

        # ── Safety: ensure valve starts closed ────────────────────────────────
        print("Initialising — setting valve CLOSED (0.0 V)...")
        ljm.eWriteName(handle, OUTPUT_REGISTER, V_CLOSED)
        time.sleep(2.0)
        
        # ── Open ──────────────────────────────────────────────────────────────
        print(f"\nRamping valve OPEN ({V_CLOSED} V → {V_OPEN} V)...")
        ramp(handle, V_CLOSED, V_OPEN, RAMP_STEPS, RAMP_DELAY_S)
        print(f"Valve fully open — holding for {HOLD_TIME_S} seconds...")
        time.sleep(HOLD_TIME_S)

        # ── Close ─────────────────────────────────────────────────────────────
        print(f"\nRamping valve CLOSED ({V_OPEN} V → {V_CLOSED} V)...")
        ramp(handle, V_OPEN, V_CLOSED, RAMP_STEPS, RAMP_DELAY_S)
        print("Valve fully closed.")

    except ljm.LJMError as e:
        print(f"\nLabJack error: {e}")

    except KeyboardInterrupt:
        print("\nTest interrupted by user.")
        
    finally:
        # ── Always close the valve and disconnect cleanly ─────────────────────
        if handle is not None:
            print("\nSafety close — setting DAC0 to 0.0 V before disconnecting...")
            ljm.eWriteName(handle, OUTPUT_REGISTER, 0.0)
            ljm.close(handle)
            print("LabJack disconnected.")


if __name__ == "__main__":
    main()

Connected to LabJack T7 [Serial: 470042305]
Output register: DAC1
----------------------------------------
Initialising — setting valve CLOSED (0.0 V)...

Ramping valve OPEN (0.0 V → 5.0 V)...
  DAC0 = 0.00 V
  DAC0 = 0.50 V
  DAC0 = 1.00 V
  DAC0 = 1.50 V
  DAC0 = 2.00 V
  DAC0 = 2.50 V
  DAC0 = 3.00 V
  DAC0 = 3.50 V
  DAC0 = 4.00 V
  DAC0 = 4.50 V
  DAC0 = 5.00 V
Valve fully open — holding for 120 seconds...

Test interrupted by user.

Safety close — setting DAC0 to 0.0 V before disconnecting...
LabJack disconnected.


In [ ]:
# Register for the digital I/O pin you selected
RELAY_PIN = "DIO3"  # FIO4 maps to DIO4 on the T7

def main():
    try:
        handle = ljm.openS("T7", "ANY", "ANY")
        print("Connected to LabJack T7. Starting pump cycle test...")

        while True:
            ljm.eWriteName(handle, RELAY_PIN, 1)  # Set pin HIGH (5V)
            time.sleep(5.0)

            ljm.eWriteName(handle, RELAY_PIN, 0)  # Set pin LOW (0V)
            time.sleep(5.0)

    except ljm.LJMError as e:
        print(f"LabJack Error: {e}")
    except KeyboardInterrupt:
        print("\nTest stopped by user.")
    finally:
        if 'handle' in locals():
            # Turn pump off as a safety measure before closing connection
            ljm.eWriteName(handle, RELAY_PIN, 0)
            ljm.close(handle)
            print("LabJack closed safely. Pump set to safe OFF state.")

if __name__ == "__main__":
    main()

In [8]:
from labjack import ljm
import time

# ---------------- Configuration ----------------
VALVE_REGISTER = "DAC1"
PUMP_RELAY_PIN = "DIO3"

V_CLOSED = 0.0
V_OPEN = 5.0

RAMP_STEPS = 10
RAMP_DELAY_S = 0.5

VALVE_SETTLE_TIME_S = 5.0   # Wait after valve fully opens
PUMP_RUN_TIME_S = 5.0       # Pump run duration


def ramp(handle, v_start, v_end, steps, delay_s):
    """Ramp valve position."""
    for i in range(steps + 1):
        voltage = v_start + (v_end - v_start) * (i / steps)
        ljm.eWriteName(handle, VALVE_REGISTER, voltage)
        print(f"Valve = {voltage:.2f} V")
        time.sleep(delay_s)


def main():
    handle = None

    try:
        # Connect to T7
        handle = ljm.openS("T7", "ANY", "ANY")
        info = ljm.getHandleInfo(handle)

        print(f"Connected to T7 [Serial: {info[2]}]")
        print("-" * 40)

        # Safety startup state
        print("Setting safe startup state...")
        ljm.eWriteName(handle, VALVE_REGISTER, V_CLOSED)
        ljm.eWriteName(handle, PUMP_RELAY_PIN, 0)

        time.sleep(2)

        # Open valve
        print("\nOpening valve...")
        ramp(
            handle,
            V_CLOSED,
            V_OPEN,
            RAMP_STEPS,
            RAMP_DELAY_S
        )

        print(f"Waiting {VALVE_SETTLE_TIME_S} s for valve to settle...")
        time.sleep(VALVE_SETTLE_TIME_S)

        # Start pump
        print("\nTurning pump ON...")
        ljm.eWriteName(handle, PUMP_RELAY_PIN, 1)

        print(f"Pump running for {PUMP_RUN_TIME_S} seconds...")
        time.sleep(PUMP_RUN_TIME_S)

        # Stop pump
        print("\nTurning pump OFF...")
        ljm.eWriteName(handle, PUMP_RELAY_PIN, 0)

        # Close valve
        print("\nClosing valve...")
        ramp(
            handle,
            V_OPEN,
            V_CLOSED,
            RAMP_STEPS,
            RAMP_DELAY_S
        )

        print("Cycle complete.")

    except ljm.LJMError as e:
        print(f"LabJack Error: {e}")

    except KeyboardInterrupt:
        print("\nInterrupted by user.")

    finally:
        if handle is not None:
            print("\nApplying safe shutdown state...")
            ljm.eWriteName(handle, PUMP_RELAY_PIN, 0)
            ljm.eWriteName(handle, VALVE_REGISTER, V_CLOSED)

            ljm.close(handle)
            print("LabJack disconnected.")


if __name__ == "__main__":
    main()

Connected to T7 [Serial: 470042305]
----------------------------------------
Setting safe startup state...

Opening valve...
Valve = 0.00 V
Valve = 0.50 V
Valve = 1.00 V
Valve = 1.50 V
Valve = 2.00 V
Valve = 2.50 V
Valve = 3.00 V
Valve = 3.50 V
Valve = 4.00 V
Valve = 4.50 V
Valve = 5.00 V
Waiting 5.0 s for valve to settle...

Turning pump ON...
Pump running for 5.0 seconds...

Turning pump OFF...

Closing valve...
Valve = 5.00 V
Valve = 4.50 V
Valve = 4.00 V
Valve = 3.50 V
Valve = 3.00 V
Valve = 2.50 V
Valve = 2.00 V
Valve = 1.50 V
Valve = 1.00 V
Valve = 0.50 V
Valve = 0.00 V
Cycle complete.

Applying safe shutdown state...
LabJack disconnected.
